# Import Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.linear_model import LinearRegression

# Read Data

In [3]:
trainDf = pd.read_csv("Dataset/train_clinical_data.csv")
print(trainDf.shape)
trainDf.head()

(2615, 8)


,visit_id,patient_id,visit_month,updrs_1,updrs_2,updrs_3,updrs_4,upd23b_clinical_state_on_medication
0,55_0,55,0,10.0,6.0,15.0,NaN,NaN
1,55_3,55,3,10.0,7.0,25.0,NaN,NaN
2,55_6,55,6,8.0,10.0,34.0,NaN,NaN
3,55_9,55,9,8.0,9.0,30.0,0.0,On
4,55_12,55,12,10.0,10.0,41.0,0.0,On


# Model Training

In [5]:
model = {}
target = ["updrs_1", "updrs_2", "updrs_3", "updrs_4"]

for u in target:
        
    # Drop NAs
    temp = trainDf.dropna(subset=[u]) 
    
    # Train data
    X = temp['visit_month']
    y = temp[u]
        
    trained = LinearRegression().fit(X.values.reshape(-1, 1), y)
    
    # Save model
    model[u] = trained

In [7]:
def get_predictions(my_train, pro, model):

    # Forecast
    my_train = my_train.fillna(0)
    
    for u in target:
        
        # Here is where we will save the final results
        my_train['result_' + str(u)] = 0
  
        # Predict    
        X = my_train["visit_month"]
        
        if u == 'updrs_4':
            my_train['result_' + str(u)] = 0
        else:
            my_train['result_' + str(u)] = np.ceil(model[u].predict(X.values.reshape(-1, 1)))

        
    # Format for final submission
    result = pd.DataFrame()

    for m in [0, 6, 12, 24]:
        for u in [1, 2, 3, 4]:

            temp = my_train[["visit_id", "result_updrs_" + str(u)]]
            temp["prediction_id"] = temp["visit_id"] + "_updrs_" + str(u) + "_plus_" + str(m) + "_months"
            temp["rating"] = temp["result_updrs_" + str(u)]
            temp = temp [['prediction_id', 'rating']]

            result = result.append(temp)            
    result = result.drop_duplicates(subset=['prediction_id', 'rating'])

    return result



In [9]:
# Run once to check results
get_predictions(trainDf, None, model)

C:\Users\sport\AppData\Local\Temp\ipykernel_29424\2200218608.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp["prediction_id"] = temp["visit_id"] + "_updrs_" + str(u) + "_plus_" + str(m) + "_months"
C:\Users\sport\AppData\Local\Temp\ipykernel_29424\2200218608.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp["rating"] = temp["result_updrs_" + str(u)]
C:\Users\sport\AppData\Local\Temp\ipykernel_29424\2200218608.py:31: FutureWarning: The frame.append method is deprecated and will be removed f

,prediction_id,rating
0,55_0_updrs_1_plus_0_months,7.0
1,55_3_updrs_1_plus_0_months,7.0
2,55_6_updrs_1_plus_0_months,7.0
3,55_9_updrs_1_plus_0_months,7.0
4,55_12_updrs_1_plus_0_months,7.0
...,...,...
2610,65043_48_updrs_4_plus_24_months,0.0
2611,65043_54_updrs_4_plus_24_months,0.0
2612,65043_60_updrs_4_plus_24_months,0.0
2613,65043_72_updrs_4_plus_24_months,0.0
